# KẺ MẠO DANH

Notebook baseline tạo trực tiếp `submission.zip` cho tập kiểm tra.

## 1. Cấu hình

In [ ]:
import os

# Phải đặt trước khi torch khởi tạo CUDA.
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

from pathlib import Path
import random
import time
import zipfile

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

DATA_ROOT = Path('/home/user/TACVU1/data')
SPLIT = 'public_test'        # đổi thành 'private_test' ở giai đoạn kiểm tra bí mật

EPOCHS = 3
IMAGE_SIZE = 64
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Cố định kết quả giữa các lần chạy: cùng seed -> cùng file nộp.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[cấu hình] split={SPLIT} | epochs={EPOCHS} | batch={BATCH_SIZE} | ảnh={IMAGE_SIZE}px | seed={SEED}')
print(f'[cấu hình] thiết bị: {DEVICE}'
      + (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))

## 2. Mô hình và phép biến đổi ảnh

In [ ]:
class CNN2(nn.Module):
    """CNN hai khối tích chập, phân loại ảnh thật/giả."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


normalize = transforms.Normalize((.485, .456, .406), (.229, .224, .225))

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])

eval_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    normalize,
])

print(f'[mô hình] CNN2: {sum(p.numel() for p in CNN2().parameters()):,} tham số')

## 3. Dataset

In [ ]:
def load_image(path: Path, tf):
    with Image.open(path) as image:
        return tf(ImageOps.exif_transpose(image).convert('RGB'))


class SingleImages(Dataset):
    """Tách mỗi cặp train thành hai mẫu ảnh đơn kèm nhãn thật/giả."""

    def __init__(self, pairs, tf):
        self.tf = tf
        self.rows = []
        for row in pairs.itertuples(index=False):
            self.rows.append((row.image_0, int(row.fake_position == 0)))
            self.rows.append((row.image_1, int(row.fake_position == 1)))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        relative, label = self.rows[index]
        return load_image(DATA_ROOT / 'train' / relative, self.tf), label


class ImagePairs(Dataset):
    """Trả về cả hai ảnh của một cặp để so sánh khi dự đoán."""

    def __init__(self, pairs):
        self.pairs = pairs.reset_index(drop=True)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        row = self.pairs.iloc[index]
        first = load_image(DATA_ROOT / SPLIT / row.image_0, eval_tf)
        second = load_image(DATA_ROOT / SPLIT / row.image_1, eval_tf)
        return first, second, index


print('[dataset] đã định nghĩa SingleImages (train) và ImagePairs (dự đoán)')

## 4. Nạp dữ liệu

In [ ]:
train_pairs = pd.read_csv(DATA_ROOT / 'train' / 'pairs.csv', dtype={'pair_id': str})
query = pd.read_csv(DATA_ROOT / SPLIT / 'pairs.csv', dtype={'pair_id': str})

train_set = SingleImages(train_pairs, train_tf)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)

print(f'[dữ liệu] train: {len(train_pairs):,} cặp -> {len(train_set):,} ảnh')
print(f'[dữ liệu] {SPLIT}: {len(query):,} cặp cần dự đoán')
print(f'[dữ liệu] phân bố nhãn train: {train_pairs.fake_position.value_counts().to_dict()}')

## 5. Huấn luyện

In [ ]:
model = CNN2().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(1, EPOCHS + 1):
    model.train()
    started = time.time()
    total_loss = 0.0
    correct = 0
    seen = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        seen += labels.size(0)

    print(f'[huấn luyện] epoch {epoch}/{EPOCHS} | loss={total_loss / seen:.4f} '
          f'| accuracy={correct / seen:.4f} | {time.time() - started:.1f}s')

print('[huấn luyện] hoàn tất')

## 6. Dự đoán và đóng gói bài nộp

In [ ]:
model.eval()
predictions = np.zeros(len(query), dtype=np.int64)
started = time.time()

with torch.inference_mode():
    for first, second, indices in DataLoader(ImagePairs(query), batch_size=BATCH_SIZE):
        fake_prob_0 = torch.softmax(model(first.to(DEVICE)), 1)[:, 1]
        fake_prob_1 = torch.softmax(model(second.to(DEVICE)), 1)[:, 1]
        predictions[indices.numpy()] = (fake_prob_1 > fake_prob_0).cpu().numpy()

submission = pd.DataFrame({'pair_id': query.pair_id, 'fake_position': predictions})
submission.to_csv('submission.csv', index=False)

with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write('submission.csv')

print(f'[dự đoán] {len(submission):,} cặp trong {time.time() - started:.1f}s')
print(f'[dự đoán] phân bố nhãn: {submission.fake_position.value_counts().to_dict()}')
print('[nộp bài] đã tạo submission.zip (chứa đúng submission.csv ở thư mục gốc)')